In [ ]:
OUTPUT_LATEX=True # set to True if you want to export LaTeX images, otherwise defaulting to png

In [ ]:
from local_paths import MASTER_DATA_PATH
from repo_paths import *
import matplotlib
if OUTPUT_LATEX:
    matplotlib.use('pgf')

import matplotlib.pyplot as plt # plt import has to be after matplotlib.use('pgf')
plt.rcParams.update({
    'font.family': 'serif',
    'text.usetex': True,
    'pgf.rcfonts': False,
    'font.size': 8
})

import seaborn as sns
import pandas as pd
import numpy as np
import scipy
import statsmodels
from scipy.stats import pearsonr, iqr
from statsmodels.iolib.summary import Summary

In [ ]:
def output(label,p):
    if isinstance(p, matplotlib.axes.Axes):
        p = p.get_figure()

    if isinstance(p, matplotlib.figure.Figure) or isinstance(p, sns.FacetGrid):
        if OUTPUT_LATEX:
            p.savefig(f'{LATEX_PATH}/fig_{label}.pgf', bbox_inches='tight')
        else:
            display(p)
            p.savefig(f'{OUTPUT_PATH}/fig_{label}.png', bbox_inches='tight')
        plt.close()
    elif isinstance(p, Summary):
        if OUTPUT_LATEX:
            with open(f'{LATEX_PATH}/tab_ols_{label}.tex', 'w') as f:
                s = p.as_latex().replace('\\begin{center}','').replace('\\end{center}','') #for some reason the center environment causes a parsing error, so removing it
                f.write(s)
        else:
            display(p)
    else:
        raise TypeError('p is not Axes, Figure, or Summary: '+type(p).__name__)

In [ ]:
LATEX_TEXT_W = 345 #in pt
FIG_W  = (345/72.27) #pyplot works in inches
GR = (1+ 5**.5)/2 # golden ratio
FIG_H = FIG_W / GR
FIGSIZE_SINGLE = (FIG_W,FIG_H)

In [ ]:
#throughout the notebook key value pairs can be stored here for later use in the latex report
keyvals = {}

import sys
keyvals['vpython'] = sys.version.split(' ') [0]
keyvals['vpandas'] = pd.__version__
keyvals['vscipy'] = scipy.__version__
keyvals['vstatsmodels'] = statsmodels.__version__
keyvals['vseaborn'] = sns.__version__

In [ ]:
import os
from pathlib import Path
Path(OUTPUT_PATH).mkdir(parents=True, exist_ok=True)

participants = pd.read_csv(f'{MASTER_DATA_PATH}/{PARTICIPANTS_FNAME}.csv', index_col='subject_id')
participants.ASEP = participants.ASEP * 100 #TODO: move to preprocessing

In [ ]:
def get_stats(series):
    min_val = series.min()
    max_val = series.max()
    return {
        'mean': round(series.mean(), 1),
        'median': round(series.median(), 1),
        'min': round(min_val, 1),
        'max': round(max_val, 1),
        'range': int(round(max_val - min_val, 0))
    }

# Descriptive statistics for ASEP
keyvals.update({f'asep{key}': val for key, val in get_stats(participants['ASEP']).items()})

# Descriptives and counts for groups
subgroups = {
    'male': participants[participants['Sex'] == 'Male'],
    'female': participants[participants['Sex'] == 'Female'], 
    'vegetarian': participants[participants['vegetarian'] == 1],
    'omnivore': participants[participants['vegetarian'] == 0]
}

for group_name, group_data in subgroups.items():
    keyvals.update({f'asep{group_name}{key}': val for key, val in get_stats(group_data['ASEP']).items()})
    keyvals[f'n{group_name}'] = len(group_data)

In [ ]:
import re

def tex_escape(text):
    if not isinstance(text,str):
        text = str(text)
    
    conv = {
        '&': r'\&',
        '%': r'\%',
        '$': r'\$',
        '#': r'\#',
        '_': r'\_',
        '{': r'\{',
        '}': r'\}',
        '~': r'\textasciitilde{}',
        '^': r'\^{}',
        '\\': r'\textbackslash{}',
        '<': r'\textless{}',
        '>': r'\textgreater{}',
    }
    regex = re.compile('|'.join(re.escape(str(key)) for key in sorted(conv.keys(), key = lambda item: - len(item))))
    return regex.sub(lambda match: conv[match.group()], text)

def default_row_to_str(i,row):
    return i+' & ' + ' & '.join(map(str,row))

def write_latex_tabular_rows(label,df,row_to_str=default_row_to_str):
    with open(f'{LATEX_PATH}/rows_{label}.tex','w') as f:
        for i, row in df.iterrows():
            f.write(row_to_str(i,row))
            #circumventing an off latex bug: leaving the last row without the double backticks and using them outside the \input
            if i != df.index[-1]:
                f.write(' \\\\\n')

In [ ]:
participants['ep-total-carbs'] = participants['ep-CHO'] + participants['ep-FIBC']

In [ ]:
#Table Median nutrient intakes by sex

INDENTED_NUTRIENTS = {
    'ep-SUGAR', 'ep-SUCS', 'FIBC_per_d',  # Carbohydrates
    'ep-FASAT', 'ep-FAMS', 'ep-FAPU', 'ep-FATRN', 'CHOL_per_d',  # Fat
    'PROT_g_per_BW',  # Protein
    'ALC_per_d'  # Alcohol
}

# Rounding rules
ROUNDING_RULES = {
    'Protein, g/kg body weight': 2,
    'Energy, kJ/d': 0,
    'Cholesterol, mg/d': 0,
    'Sodium, g/d': 0,
}
DEFAULT_DECIMALS = 1

intake_cols_and_labels = {
    'ENERJ_per_d': 'Energy, kJ/d',
    'ep-total-carbs': 'Carbohydrates, E%',
    'ep-SUGAR': 'Sugars, E%',
    'ep-SUCS': 'Sucrose, E%',
    'FIBC_per_d': 'Fibre, g/d',
    'ep-FAT': 'Fat, E%',
    'ep-FASAT': 'Saturated fatty acids, E%',
    'ep-FAMS': 'Monounsaturated fatty acids, E%',
    'ep-FAPU': 'Polyunsaturated fatty acids, E%',
    'ep-FATRN': 'Trans fatty acids, E%',
    'CHOL_per_d': 'Cholesterol, mg/d',
    'ep-PROT': 'Protein, E%',
    'PROT_g_per_BW': 'Protein, g/kg body weight',
    'ep-ALC': 'Alcohol, E%',     
    'ALC_per_d': 'Alcohol, g/d',
    'NA_per_d': 'Sodium, g/d',
}

label_to_key = {v: k for k, v in intake_cols_and_labels.items()}

median_intakes_by_sex = participants[
    [*intake_cols_and_labels.keys(),'Sex']
].groupby('Sex').agg(
    ['median', ('Q1',lambda x: x.quantile(0.25)), ('Q3',lambda x: x.quantile(0.75))]
).T.unstack()
median_intakes_by_sex.index = median_intakes_by_sex.index.map(intake_cols_and_labels)

def intake_rows_to_str(i, row):

    decimals = ROUNDING_RULES.get(i, DEFAULT_DECIMALS)
    row_out = row.round(decimals)
    if decimals == 0:
        row_out = row_out.astype(int)
    
    label = tex_escape(i)
    if label_to_key[i] in INDENTED_NUTRIENTS:
        label = r'\quad ' + label
    
    sf, sm = [f'{row_out[sex, "median"]} & ({row_out[sex, "Q1"]}--{row_out[sex, "Q3"]})' 
              for sex in ['Female', 'Male']]
    
    return ' & '.join([label, sf, sm])

write_latex_tabular_rows('median_intakes_by_sex', median_intakes_by_sex, row_to_str=intake_rows_to_str)

In [ ]:
#mention median of alcohol
m, q1, q3 = participants['ALC_per_d'].quantile([0.5,0.25,0.75])
keyvals['alcgperdmedianiqr'] = f'{m:.2f} ({q1:.2f}-{q3:.2f})'

In [ ]:
serum_lipids = {
    'v02_fs_kol_mean': 'Total cholesterol, mmol/L',
    'v02_fs_kol_ldl_mean': 'LDL-cholesterol, mmol/L',
    'v02_fs_kol_hdl_mean': 'HDL-cholesterol, mmol/L',
    'v02_fs_trigly_mean': 'Triglycerides, mmol/L',
}

cholesterol_vars = ['v02_fs_kol_mean', 'v02_fs_kol_ldl_mean', 'v02_fs_kol_hdl_mean']
trigly_var = 'v02_fs_trigly_mean'

def calculate_lipid_stats(data):
    # Cholesterol: mean (SD)
    chol_stats = data[cholesterol_vars].agg(['mean', 'std']).T
    chol_stats['Concentration'] = chol_stats.apply(
        lambda row: f"{row['mean']:.1f} ({row['std']:.1f})", axis=1
    )
    
    # Triglycerides: median [Q1, Q3]
    trigly = data[trigly_var]
    trigly_stats = pd.DataFrame({
        'Concentration': [f"{trigly.median():.1f} [{trigly.quantile(0.25):.1f}, {trigly.quantile(0.75):.1f}]"]
    }, index=[trigly_var])
    
    return pd.concat([chol_stats[['Concentration']], trigly_stats])

# Calculate for all groups
groups = {
    'Full sample': participants,
    'Women': participants[participants['Sex'] == 'Female'],
    'Men': participants[participants['Sex'] == 'Male']
}

p = pd.concat([
    calculate_lipid_stats(data).rename(columns={'Concentration': name})
    for name, data in groups.items()
], axis=1)

p.index = p.index.map(serum_lipids)

p.to_latex(
    f'{LATEX_PATH}/tab_median_serum_lipids.tex',
    column_format='lrrr',
    escape=False
)

In [ ]:
n = len(participants.index)
keyvals['nparticipant'] = n

nfemale = participants.Sex.value_counts()['Female']
keyvals['nfemale'] = nfemale
keyvals['nmale'] = keyvals['nparticipant'] - keyvals['nfemale']
keyvals['pctfemale'] = round(nfemale/n *100)

nv = participants.vegetarian.sum()
keyvals['nvegetarian'] = nv
keyvals['nomnivore'] = n-nv
keyvals['pctvegetarian'] = round(nv/n *100)
keyvals['pctomnivore'] = 100 - keyvals['pctvegetarian']

keyvals['nnotmissingsmoking'] = participants['var26'].notna().sum() 
keyvals['nsmokers'] = (participants['var26'] == 2).sum() #2:smokers, 1:less than weekly, 0:non-smokers

keyvals['nalcoholuser'] = (participants['ep-ALC'] > 0).sum() # n alcohol-users
keyvals['pctalcoholuser'] = round(keyvals['nalcoholuser']/n *100) # % alcohol-users
keyvals['nalcoholnonuser'] = (participants['ep-ALC'] == 0).sum() #counting alcohol-non-users

keyvals['agemedian'] = round(participants.v01_agecon.median())
keyvals['ageqoneqthree'] = f"{round(participants.v01_agecon.quantile(0.25))}–{round(participants.v01_agecon.quantile(0.75))}"

keyvals['pctsmokers'] = round((participants['var26'] == 2).mean() * 100)
keyvals['nnotmissingedu'] = participants['Educational Attainment Level'].notna().sum()

keyvals['nmissingactivity'] = participants['v02_mvpa'].isnull().sum()
keyvals['nnotmissingactivity'] = participants['v02_mvpa'].notna().sum()

keyvals['frdaysfour'] = len(participants[participants['fr_days'] == 4])
keyvals['frdaysthree'] = len(participants[participants['fr_days'] == 3])

keyvals['nmissingldl'] = participants['v02_fs_kol_ldl_mean'].isnull().sum()
keyvals['nmissinghdl'] = participants['v02_fs_kol_hdl_mean'].isnull().sum()
keyvals['nmissingtrigly'] = participants['v02_fs_trigly_mean'].isnull().sum()
keyvals['notmissingldl'] = participants['v02_fs_kol_ldl_mean'].notna().sum()
keyvals['notmissingtrigly'] = participants['v02_fs_trigly_mean'].notna().sum()
keyvals['notmissingldl_men'] = participants[participants['Sex'] == 'Male']['v02_fs_kol_ldl_mean'].notna().sum()
keyvals['notmissingtrigly_men'] = participants[participants['Sex'] == 'Male']['v02_fs_trigly_mean'].notna().sum()
keyvals['notmissingldl_women'] = participants[participants['Sex'] == 'Female']['v02_fs_kol_ldl_mean'].notna().sum()
keyvals['notmissingtrigly_women'] = participants[participants['Sex'] == 'Female']['v02_fs_trigly_mean'].notna().sum()

keyvals['nnotmissingsmoking'] = participants['var26'].notna().sum() #adding n for non-missing smoking status

formats = {
    'v01_agecon': '{:.0f}',
    'BMI': '{:.1f}',
    'v02_mvpa': '{:.0f}'
}
labels = [
    'Age ($y$)',
    'BMI ($kg/m^2$)',
    'Physical Activity ($min/d$)',
]

df = participants[formats.keys()].agg(lambda x: x.quantile([0.5, 0.25, 0.75]))
for col, fmt in formats.items(): df[col] = df[col].map(fmt.format)
df = df.T
df['q1-q3'] = df.apply(lambda row: f"[{row[0.25]}--{row[0.75]}]", axis=1)
df = df.drop(columns=[0.25, 0.75])
df.index = labels
write_latex_tabular_rows('age_bmi_activity', df.round(0))

cm_counts = participants.cm_intensity.value_counts().to_frame()
cm_counts.loc['No statin medication'] = participants.cm_intensity.isna().sum()
cm_counts.columns = ['Count']
cm_counts.index.name = 'Intensity'
cm_counts['Percentage'] = cm_counts.Count / n * 100

desired_order = ['High', 'Moderate', 'Low', 'No statin medication']
cm_counts = cm_counts.reindex(desired_order)

write_latex_tabular_rows('cm_intensity',cm_counts.round().astype(int))

keyvals['statin_none'] = cm_counts.loc['No statin medication','Count']
keyvals['statin_none_pct'] = cm_counts.loc['No statin medication','Percentage']
keyvals['statin_any'] = n - keyvals['statin_none']
keyvals['statin_any_pct'] = round(100 - keyvals['statin_none_pct'])

edu = participants['Educational Attainment Level'].value_counts().to_frame()
edu.columns = ['Count']
edu.index.name = 'Educational Attainment Level'
edu['Percentage'] = edu.Count / n * 100

write_latex_tabular_rows('education',edu.round().astype(int))

In [ ]:
#histogram of asep distribution
bins = np.arange(10, 71, 5)

fig, ax = plt.subplots(1, 1, figsize=FIGSIZE_SINGLE)
sns.histplot(ax=ax, data=participants, x='ASEP', hue='Sex', bins=bins, multiple="layer", alpha=0.6)
legend = ax.get_legend()
if legend:
    replacements = {'Female': 'Women', 'Male': 'Men'}
    for text in legend.get_texts():
        text.set_text(replacements.get(text.get_text(), text.get_text()))
    legend.set_title('Sex')
ax.set_xlabel(r'ASEP (\%)')
ax.set_ylabel("Count")
ax.set_xticks(bins[::2])
output('hist', fig)

In [ ]:
#todo female-> Women, male -> Men. o -> Omnivore, 1 -> Vegetarian, ASEP-label with (%), save figure to be used in LaTeX
ax= sns.boxplot(data=participants, x='Sex', y='ASEP', fliersize=0, boxprops={'facecolor': '0.8', 'edgecolor': 'black', 'alpha': 0.5})
sns.stripplot(ax=ax, data=participants, x='Sex', y='ASEP', hue= 'vegetarian', s=5)
plt.xlabel('')
plt.ylabel('ASEP')
plt.legend(title='')
plt.show()

In [ ]:
intake_rows = pd.read_csv(f'{MASTER_DATA_PATH}/{INTAKES_FNAME}.csv', low_memory=False)

fao_subgroups = pd.read_csv(f'{DATA_PATH}/fao_groups.csv', index_col='subgroup_code')
own_subgroups = pd.read_csv(f'{DATA_PATH}/own_groups.csv', index_col='subgroup_code')
subgroups = pd.concat([fao_subgroups, own_subgroups])
if not subgroups.index.is_unique:
    raise ValueError('Check food group files, duplicated subgroup_code')

subgroups['group_name_short'] = subgroups['group_name_short'].replace('Milk and milk products', 'Milk and dairy')

intake_rows['group_code'] = intake_rows['fao_subgroup_code'].map(subgroups['group_code'])

In [ ]:
n_all = intake_rows['code'].nunique()
n_plant = intake_rows[intake_rows['animal_proportion'] == 0]['code'].nunique()
n_animal = intake_rows[intake_rows['animal_proportion'] == 1]['code'].nunique()
n_mixed = n_all-n_plant-n_animal

keyvals['nfoodcodes'] = n_all
keyvals['nfoodcodesplant'] = n_plant
keyvals['pctfoodcodesplant'] = round(n_plant / n_all*100)
keyvals['nfoodcodesanimal'] = n_animal
keyvals['pctfoodcodesanimal'] = round(n_animal / n_all*100)
keyvals['nfoodcodesmixed'] = n_mixed
keyvals['pctfoodcodesmixed'] = round(n_mixed / n_all*100)

In [ ]:
# bar plots of different food groups proportional contribution to energy intake
group_names = subgroups[['group_code','group_name_short']].drop_duplicates().set_index('group_code')
e_per_group = intake_rows[['group_code','ENERJ','ENERJ_animal']].groupby('group_code').sum().join(group_names)

e_per_group['ENERJ_plant'] = e_per_group.ENERJ - e_per_group.ENERJ_animal
e_per_group['e_animal_pros'] = e_per_group.ENERJ_animal / e_per_group.ENERJ_animal.sum() * 100
e_per_group['e_plant_pros'] = e_per_group.ENERJ_plant / e_per_group.ENERJ_plant.sum() * 100

def ingredient_group_bars(figname, x):
    # Set x-axis label based on energy type
    if x == 'e_animal_pros':
        xlabel = 'Contribution to animal source energy (%)'
    elif x == 'e_plant_pros':
        xlabel = 'Contribution to plant source energy (%)'

    mask = (e_per_group[x] > 1).values
    data = e_per_group[mask].sort_values(by=x)
    w = FIGSIZE_SINGLE[0] * .9
    h = FIGSIZE_SINGLE[0] / 20 * len(data.index)
    fig, ax = plt.subplots(1, 1, figsize=(w,h))
    sns.barplot(
        data=data,
        x=x,
        y='group_name_short',
        hue='group_name_short',
        palette='viridis'
    )
    ax.set_ylabel(None)
    ax.set_xlabel(xlabel)
    plt.subplots_adjust(left=0.15)

    output(figname, fig)

keyvals['nfoodsubgroups'] = len(subgroups.index)

#Animal energy was mainly derived from milk and dairy foods and meat and meat products, which respectively provided X\% and X\% of the total energy from \ac{ASF}. Fish and seafood use was low.
keyvals['dairyeanimalpros'] = e_per_group.loc[e_per_group['group_name_short'] == 'Milk and dairy', 'e_animal_pros'].iloc[0].round(1)
keyvals['meateanimalpros'] = e_per_group.loc[e_per_group['group_name_short'] == 'Meat', 'e_animal_pros'].iloc[0].round(1)


e_fish = e_per_group.loc[e_per_group['group_name_short'] == 'Fish', 'ENERJ'].iloc[0]
keyvals['fishepros'] = (e_fish / e_per_group.ENERJ.sum() * 100).round(1)

In [ ]:
ingredient_group_bars('bar_animal','e_animal_pros')

In [ ]:
ingredient_group_bars('bar_plant','e_plant_pros')

In [ ]:
from statsmodels.stats.multitest import fdrcorrection

def write_corr_table(label,corr_df):
    corr_df['fdr_sign'],_ = fdrcorrection(corr_df['p_value'])

    def p_str(row):
        p = row['p_value']
        ps = '\\textless{} 0.001' if p < 0.001 else f'{p:.3f}'
        ps += '*' if row['fdr_sign'] else '\\hphantom{*}'
        return ps
    
    corr_df['p_str'] = corr_df.apply(p_str, axis=1)
    corr_out = corr_df[['rho','p_str']]
    corr_out.columns = ['rho','p']
    corr_out.index = corr_out.index.map(tex_escape)

    corr_out.to_latex(
        f'{LATEX_PATH}/tab_{label}.tex',
        header = ['$r_{{s}}$','$p$'],
        column_format='lrr',
        float_format='%.2f', #only applies to the r value, p's are already strings
        escape=False
    )

In [ ]:
from scipy.stats import spearmanr

intake_metrics = {
    'ep-total-carbs': 'Carbohydrates, E%',
    'ep-PROT': 'Protein, E%', 
    'ep-FAT': 'Fat, E%',
    'ep-FASAT': 'SAFA, E%',
    'ep-FAPU': 'PUFA, E%', 
    'ep-FAMS': 'MUFA, E%',
    'ep-FATRN': 'Trans fat, E%',
    'FIBC_per_MJ': 'Fibre, g/MJ',
    'CHOL_per_MJ': 'Cholesterol, mg/MJ',
    'NA_per_MJ': 'Sodium, g/MJ',
    'ENERJ_per_d': 'Energy, KJ/d',
    'ep-SUGAR': 'Sugars, E%',
    'ep-SUCS': 'Sucrose, E%',
    'ep-ALC': 'Alcohol, E%',
}
keyvals['ncorrintake'] = len(intake_metrics)

intake_corr_dict = {}
for k,v in intake_metrics.items():
    rho = spearmanr(participants.ASEP, participants[k])
    intake_corr_dict[v] = rho
    keyvals[f'corrrhoasep{k}'] = round(rho.correlation,2)
    keyvals[f'corrpasep{k}'] = round(rho.pvalue,3)
    keyvals[f'corrasep{k}'] = f'r={rho.correlation:.2f}, p={rho.pvalue:.3f}'

intake_corr_df = pd.DataFrame(intake_corr_dict).T
intake_corr_df.columns = ['rho','p_value']
intake_corr_df.sort_values(by='rho', ascending=False, inplace=True)
write_corr_table('corr_intakes',intake_corr_df)

In [ ]:
#ASEP nutrient intakes scatter plots

recommendations = {
    'ep-FAT': [25,40],
    'ep-PROT': [10,20],
    'ep-FASAT': [10],
    'ep-FAPU': [5,10],
    'FIBC_per_MJ': [3],
    'FOL_per_d': [330],
    'ep-total-carbs': [45,60]
}

def grid_of_scatterplots(endog_dict):
    
    from math import ceil
    l = len(endog_dict)
    nrows = ceil(l/2)

    fig, axs = plt.subplots(nrows,2,figsize=(FIG_W,nrows*FIG_H*0.78))
    axs = list(axs.flatten())
    
    for k,v in endog_dict.items():
        ax = axs.pop(0)
        sns.scatterplot(
            ax=ax,
            data=participants,
            x='ASEP',
            y=k,
            s=12
        )
        ax.set_ylabel(v, fontsize=8)
        ax.set_xlabel('ASEP (%)')
        ax.tick_params(axis='both', labelsize=8)
        for r in recommendations.get(k,[]):
            ax.axhline(y=r,c='red',ls=':')

        rho, p = spearmanr(participants['ASEP'], participants[k])

        x_pos = 0.05 if rho > 0 else 0.95
        h_align = 'left' if rho > 0 else 'right'
        ax.text(
            x_pos,
            0.95,
            f'$r_{{s}}={rho:.2f}$',
            transform=ax.transAxes,
            horizontalalignment=h_align,
            verticalalignment='top'
        )

    for ax in axs: fig.delaxes(ax)
    plt.tight_layout()

    return fig

In [ ]:
fig = grid_of_scatterplots({
    'ep-total-carbs': 'Carbohydrates, E%',
    'ep-FAT': 'Fat, E%',
    'ep-PROT': 'Protein, E%',
    'FIBC_per_MJ': 'Fibre, g/MJ',
})
output('grid_macros',fig)

In [ ]:
fig = grid_of_scatterplots({
    'ep-FASAT': 'SAFA, E%',
    'ep-FAPU': 'PUFA, E%',
    'ep-FATRN': 'Trans Fatty Acids, E%',
    'CHOL_per_d': 'Cholesterol, mg/d',
})
output('grid_fats',fig)

In [ ]:
from statsmodels.formula.api import ols

def output_ols_model(label, formula, modelset):
    model = ols(formula, data=modelset).fit()

    keyvals[f'{label}rsquaredpctadj'] = round(model.rsquared_adj*100)
    keyvals[f'{label}fpvalue'] = f'{model.f_pvalue:.3f}'
    keyvals[f'{label}asepcoef'] = f'{model.params["ASEP"]:.3f}'
    keyvals[f'{label}asepexpcoef'] = f'{np.exp(model.params["ASEP"]):.3f}' #adding a keyval for the exponentiated beta in triglyceride model 
    keyvals[f'{label}asepexpcoefinpercent'] = f'{abs((np.exp(model.params["ASEP"]) - 1) * 100):.1f}'  # Adding a keyval for the absolute exponentiated beta in triglyceride model expressed as a percentage

    keyvals[f'{label}aseppvalue'] = f'{model.pvalues["ASEP"]:.3f}'

    keyvals[f'{label}n'] = f'{model.nobs:.0f}'
    keyvals[f'{label}asepse'] = f'{model.bse["ASEP"]:.3f}'
    ci = model.conf_int().loc["ASEP"]
    keyvals[f'{label}asepci'] = f'({ci[0]:.3f}, {ci[1]:.3f})'
    keyvals[f'{label}asepcitentx'] = f'({ci[0]*10:.2f}, {ci[1]*10:.2f})'
    
    output(label, model.summary())


In [ ]:
lipid_cols = {
	'v02_fs_kol_hdl_mean': 'HDL',
	'v02_fs_kol_ldl_mean': 'LDL',
	'v02_fs_trigly_mean': 'Trigly',
}

modelset_all = participants[['ASEP','v01_agecon','BMI', 'v02_mvpa', 'var26','Sex','cm_intensity','ALC_per_d']+list(lipid_cols.keys())].copy()
modelset_all.columns = ['ASEP','Age','BMI', 'PhysicalActivity','Smoking','Sex','Statins','ALC_per_d']+list(lipid_cols.values())
modelset_all['Trigly_log'] = np.log(modelset_all['Trigly'])
modelset_all['Statins'] = modelset_all['Statins'].fillna('No').replace({'Low': 'Yes', 'High': 'Yes', 'Moderate': 'Yes'})
modelset_all['Smoking'] = modelset_all['Smoking'].map({2:'Yes',1:'No',0:'No'})


modelset_all['Age_original'] = modelset_all['Age']
modelset_all['Age'] = modelset_all['Age'] - modelset_all['Age'].mean() #mean-centering age, but keeping original for descriptive table

In [ ]:
output_ols_model(
    'hdl',
    'HDL ~  ASEP + Age * C(Sex) + BMI + PhysicalActivity + ALC_per_d + C(Statins, Treatment(reference="No")) + C(Smoking, Treatment(reference="No"))',
    modelset_all
)

In [ ]:
output_ols_model(
    'ldl',
    'LDL ~ ASEP + Age * C(Sex) + BMI + PhysicalActivity + C(Statins, Treatment(reference="No")) + ALC_per_d + C(Smoking, Treatment(reference="No"))',
    modelset_all
)

In [ ]:
output_ols_model(
    'trigly',
    'Trigly_log ~ ASEP + Age * C(Sex) + BMI + PhysicalActivity + C(Statins, Treatment(reference="No")) + ALC_per_d + C(Smoking, Treatment(reference="No"))',
    modelset_all
)

In [ ]:
#For sensitivity analysis
modelset_no_statins = modelset_all[modelset_all['Statins'] == 'No']
modelset_women = modelset_all[modelset_all['Sex'] == 'Female']

In [ ]:
output_ols_model(
    'hdl_nostatin',
    'HDL ~  ASEP + Age * C(Sex) + BMI + PhysicalActivity + ALC_per_d + C(Smoking, Treatment(reference="No"))',
    modelset_no_statins
)

In [ ]:
output_ols_model(
    'ldl_nostatin',
    'LDL ~ ASEP + Age * C(Sex) + BMI + PhysicalActivity + ALC_per_d + C(Smoking, Treatment(reference="No"))',
    modelset_no_statins
)

In [ ]:
output_ols_model(
    'trigly_nostatin',
    'Trigly_log ~ ASEP + Age * C(Sex) + BMI + PhysicalActivity + ALC_per_d + C(Smoking, Treatment(reference="No"))',
    modelset_no_statins
)

In [ ]:
output_ols_model(
    'hdl_women',
    'HDL ~ ASEP + Age + BMI + PhysicalActivity + ALC_per_d + C(Smoking, Treatment(reference="No"))+ C(Statins, Treatment(reference="No"))',
    modelset_women
)

In [ ]:
output_ols_model(
    'trigly_women',
    'Trigly_log ~ ASEP + Age + BMI + PhysicalActivity + ALC_per_d + C(Smoking, Treatment(reference="No"))+ C(Statins, Treatment(reference="No"))',
    modelset_women
)

In [ ]:
output_ols_model(
    'ldl_women',
    'LDL ~ ASEP + Age + BMI + PhysicalActivity + ALC_per_d + C(Smoking, Treatment(reference="No"))+ C(Statins, Treatment(reference="No"))',
    modelset_women
)

In [ ]:
#table for subgroup characteristics

def describe_subgroup(df):
    return {
        'n': f"{len(df)}",
        'Female, n (%)': f"{(df['Sex'] == 'Female').sum()} ({(df['Sex'] == 'Female').mean() * 100:.1f}%)",
        'Age (years), median [Q1, Q3]': f"{df['Age_original'].median():.1f} [{df['Age_original'].quantile(0.25):.1f}, {df['Age_original'].quantile(0.75):.1f}]",
        'BMI (kg/m²), median [Q1, Q3]': f"{df['BMI'].median():.1f} [{df['BMI'].quantile(0.25):.1f}, {df['BMI'].quantile(0.75):.1f}]",
        'LDL-cholesterol (mmol/L), mean (SD)': f"{df['LDL'].mean():.1f} ({df['LDL'].std():.1f})",
        'HDL-cholesterol (mmol/L), mean (SD)': f"{df['HDL'].mean():.1f} ({df['HDL'].std():.1f})",
        'Triglycerides (mmol/L), median [Q1, Q3]': f"{df['Trigly'].median():.1f} [{df['Trigly'].quantile(0.25):.1f}, {df['Trigly'].quantile(0.75):.1f}]"
    }

# Generate stats
full_stats = describe_subgroup(modelset_all)
women_stats = describe_subgroup(modelset_women)
nostatin_stats = describe_subgroup(modelset_no_statins)

# Create DataFrame with characteristics as index
df_subgroups = pd.DataFrame({
    'Full sample': full_stats,
    'Women only': women_stats,
    'Statin non-users': nostatin_stats
})

df_subgroups.to_latex(f"{LATEX_PATH}/tab_subgroup_characteristics.tex", escape=True)

In [ ]:
e_per_person = intake_rows.groupby('subject_id')['ENERJ'].sum()

check_df = participants['ENERJ'].to_frame().join(e_per_person, lsuffix='_participants',rsuffix='_intake_rows')
check_df['delta'] = check_df['ENERJ_participants'] - check_df['ENERJ_intake_rows']
divergence = check_df[check_df['delta'].abs() > 0.00001]
if len(divergence>0):
    display(divergence)
    raise ValueError('Total energy intake per person differs in two data files')

In [ ]:
#show the trend of food groups contribution to total energy intake as ASEP varies, with a loess curve

# calculate the e_pros of each ingredient group for each participant
points = (intake_rows
    .pivot_table(
        values='ENERJ',
        index='subject_id',
        columns='group_code',
        aggfunc='sum',
        fill_value=0
    )
    .div(e_per_person, axis=0)
    .reset_index()
    .melt(
        id_vars=['subject_id'],
        var_name='group_code',
        value_name='e_pros'
    )
    .merge(participants.ASEP, left_on='subject_id', right_index=True)
    .merge(group_names['group_name_short'],left_on='group_code',right_index=True)
)
points.e_pros = points.e_pros * 100

# calculate how an ingredient group's contribution to the diet (e-pros) correlates with the diet's ASEP
from scipy import stats
rho_l = []
for igc in group_names.index:
    df = points[points['group_code'] == igc]
    rho, p = stats.spearmanr(df.ASEP, df.e_pros)
        
    rho_l.append({
        'group_code': igc,
        'rho': rho,
        'abs_rho': abs(rho),
        'p_value': p,
    })

#make a dataframe of the list of correlations and print into a latex table
food_group_rho_df = pd.DataFrame(rho_l).set_index('group_code').sort_values('rho', ascending=False).dropna()


In [ ]:
import matplotlib.lines as mlines

figdata = points[
    points['group_code'].isin(food_group_rho_df[(food_group_rho_df['abs_rho']>0.2)
    & (food_group_rho_df['p_value']<0.05)].index)
]

fg = sns.lmplot(
    data=figdata,
    x='ASEP',
    y='e_pros',
    hue='group_name_short',
    lowess=True,
    height=FIG_H,
    aspect=FIG_W/FIG_H,
    scatter_kws={'s': 5,'alpha': 0.3},
    line_kws={'lw': 1.5}
)
fg.set_axis_labels(r'Participant ASEP (\%)',r'Food group prevalence in diet (E\%)')

handles, labels = fg.ax.get_legend_handles_labels()
colors = [line.get_color() for line in fg.ax.lines]

fg._legend.remove()
fg.ax.legend([mlines.Line2D([], [], color=c, linewidth=1.5) for c in colors], 
             labels, loc='upper right', frameon=True)

output('loess', fg)

In [ ]:
tab_corr_food_group = food_group_rho_df.copy()
tab_corr_food_group.index = tab_corr_food_group.index.map(group_names['group_name_short'])
tab_corr_food_group.index.name = None

write_corr_table('corr_food_group_full', tab_corr_food_group) # FDR applied here
keyvals['nfoodgroups'] = len(tab_corr_food_group) #keyval for those included in FDR

# filtering the LaTeX output table manually (excluding foods with less than 1E% contribution)
e_per_group['e_total_pros'] = e_per_group['ENERJ'] / e_per_group['ENERJ'].sum() * 100
exclude = e_per_group[e_per_group['e_total_pros'] < 1]['group_name_short']
corr_out = tab_corr_food_group[~tab_corr_food_group.index.isin(exclude)]

# writing filtered table separately
corr_out[['rho', 'p_str']].to_latex(
    f'{LATEX_PATH}/tab_corr_food_group.tex',
    header=['$r_{{s}}$', 'p'],
    column_format='lrr',
    float_format='%.2f',
    escape=False
)

In [ ]:
#check vegetables by weight separately
rho, p = stats.spearmanr(participants.ASEP, participants.veg_g)
keyvals['corrrhoasepvegg'] = round(rho,2)
keyvals['corrpasepvegg'] = round(p, 3)

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor


# Define model formulas
formulas = {
    'HDL': 'HDL ~ ASEP + Age * C(Sex) + BMI + PhysicalActivity + ALC_per_d + C(Statins, Treatment(reference="No")) + C(Smoking, Treatment(reference="No"))',
    'LDL': 'LDL ~ ASEP + Age * C(Sex) + BMI + PhysicalActivity + C(Statins, Treatment(reference="No")) + ALC_per_d + C(Smoking, Treatment(reference="No"))',
    'Trigly_log': 'Trigly_log ~ ASEP + Age * C(Sex) + BMI + PhysicalActivity + C(Statins, Treatment(reference="No")) + ALC_per_d + C(Smoking, Treatment(reference="No"))'
}

# Function to check assumptions
def check_model_assumptions(formula, data, model_name):
    print(f"\n--- Diagnostics for {model_name} ---")
    model = smf.ols(formula=formula, data=data).fit()

    # Linearity & Homoscedasticity
    fitted_vals = model.fittedvalues
    residuals = model.resid
    fig, ax = plt.subplots()
    sns.residplot(x=fitted_vals, y=residuals, lowess=True, ax=ax, line_kws={'color': 'red'})
    ax.set_title(f'Residuals vs Fitted for {model_name}')
    plt.show()

    # Normality of residuals
    sm.qqplot(residuals, line='s')
    plt.title(f'Q-Q Plot for {model_name}')
    plt.show()

    # Multicollinearity: VIFs
    exog = model.model.exog
    vif_data = pd.DataFrame()
    vif_data["VIF"] = [variance_inflation_factor(exog, i) for i in range(exog.shape[1])]
    vif_data["Variable"] = model.model.exog_names
    print("\nVariance Inflation Factors:")
    print(vif_data)

    # Condition number
    print(f"\nCondition number: {np.linalg.cond(exog)}")

# Run diagnostics for each model
for name, formula in formulas.items():
    check_model_assumptions(formula, modelset_all, name)


In [ ]:
import glob

VALS_TEX = f'{LATEX_PATH}/vals.tex'
VAL_CMD_RE = re.compile(r'\\val(\w+)')

all_tex_files = set(glob.glob(f'{LATEX_PATH}/*.tex'))
all_tex_files.discard(VALS_TEX)
keys_in_use = set()
for fpath in all_tex_files:
    with open(fpath, 'r', encoding='utf-8') as f:
        keys_in_use.update(set(re.findall(VAL_CMD_RE, f.read())))

with open(VALS_TEX, 'w') as f:
    for key, value in keyvals.items():
        clean_key = re.sub(r'[^a-zA-Z]', '', key).lower()
        f.write(f"\\newcommand{{\\val{clean_key}}}{{{value}\\xspace}}\n")
        if clean_key not in keys_in_use:
            print(f'Unused key {key}')

print(f'{len(keyvals)} keys defined, {len(keys_in_use)} used in *tex files.')